# Factor Pipeline

This is a factor working pipeline from reading to evaluating.

In [ ]:
import os
import dotenv
from pathlib import Path

import pandas as pd
import numpy as np

import quool
from parquool import Agent
from factool import DuckParquetSource, Evaluator

dotenv.load_dotenv()

## Factor Design

You are able to construction a factor by natural language, but you need to state it in a very specific way. The following block is a example. And then, you are able to copy  the definition text string to generate the factor computing code.

**NOTE:** This notebook is only available for factors that can be calculated in matrix. So if a factor needs complicated calculation or different procession for different stock, DON'T USE THIS BLOCK TO GENERATE FACTOR. Instead, you need to turn to `generate.py` for help.

## 流动性因子

### 月度换手率（monthly_turnover）

**定义**：

月度换手率定义为一个月内每日交易量的总和与每日的流通股数的比值的对数。

**计算步骤**：

1. 提取计算时点t之前的周期T=21天对应时点，记为t+T。
2. 提取数据表quotes_day中t时刻到t+T时刻对应的volume数据矩阵和circulation_a矩阵，t时刻的st和suspended矩阵。
3. 计算每t+T时间内的所有成交量总和与每日流通股数总和：
   $$
   turnover = \frac {\sum_{i=t}^{t+T} volume_i} {\sum_{i=t}^{t+T} circulation\_a_i}
   $$
4. 使用 ~st且 ~suspended数据作为掩码，过滤掉st或suspended的turnover数据，将他们设置为np.nan

In [ ]:
agent = Agent(
    instructions=Path("../docs/CODE_GENERATOR.md").read_text(encoding="utf-8")
)
prompt = ""  # Copy your full definition text here
await agent.run_streamed(prompt)

Then, you just copy generated code to a new code cell, and run it!

In [ ]:
df = ...

## Optional: Factor Saving

After generating factor, it makes things easier if you save the factor data to disk. Feel free to use `DuckParquetSource` to save any standarized data. `DuckParquetSource.save` helps you with the affairs in saving factor data. There are two available standarized data form: 1. Wide table with different stock codes for each column, and different time lables for each row; 2. Long table with two level index, the first level should be time index, and the second level should be code index. Once you prepare data in the above form, you can pass the data to `DuckParquetSource.save`. Bear in mind that if you use form 1, you need to pass another parameter called `name` to specify the name of the factor, or default name `factor` will be applied.

The `processors` parameter provide a data cleaning way before saving. If set to `None`, `zscore` and `madoutlier` with `dev=5` will be applied

In [ ]:
factor_name = ""  # Apply your factor name here
processors = None
DuckParquetSource(Path(os.getenv("FACTOR_DATA_PATH")) / factor_name).save(
    df, name=factor_name, processors=processors
)

## Factor Evaluation

`Evaluator` is an important component in `factool`. You can initialize it with simply one-line-code. Then you can apply multiple methods to evaluate the factor with different parameters. Note that factor here can be multiple, you can set different factor in a list like `[df1, df2, ...]`

In [ ]:
begin = "2015-01-01"
end = "2025-06-30"
ptype = "close_post"
horizon = 5
skip_horizon = True
ic_method = "spearman"
n_groups = 10
bucketing_mode = "single"
feasible = None
weight = None
ts_window = 252
ts_min_obs = 60
ts_intercept = True
ts_n_jobs = -1
cs_add_intercept = True
cs_cov_type = "white"
cs_white_type = "HC1"

In [ ]:
quotes_day = DuckParquetSource(Path(os.getenv("QUOTESDAY_PATH")))
price = quotes_day.get_factor(ptype, begin=begin, end=end)
low = quotes_day.get_factor("low_post", begin=begin, end=end)
high = quotes_day.get_factor("high_post", begin=begin, end=end)
limit_up = quotes_day.get_factor("limit_up", begin=begin, end=end)
limit_down = quotes_day.get_factor("limit_down", begin=begin, end=end)
feasible = (high > limit_down) & (low < limit_up)
e = Evaluator(factor=df, price=price)

In [ ]:
e.get_info_coef(horizon=horizon, skip_horizon=skip_horizon, method=ic_method)
pd.concat(
    [
        e.ic,
        e.ic.rolling(20).mean().add_suffix(" rolling 20 mean"),
        e.ic.cumsum().add_suffix(" cumsum"),
    ],
    axis=1,
).plot(
    figsize=(20, 10),
    secondary_y=(e.ic.columns + " cumsum").to_list(),
    title="IC & IC Rolling Mean & IC Cumsum",
)
e.ic.resample("YE").mean().plot.bar(figsize=(20, 10), title="Year IC Average")
display(e.ic.mean().to_frame("IC-Mean"))
display((e.ic.mean() / (e.ic.std() / np.sqrt(e.ic.shape[0]))).to_frame("IC-tvalue"))

In [ ]:
e.get_group_returns(
    n=n_groups,
    horizon=horizon,
    skip_horizon=skip_horizon,
    mode=bucketing_mode,
    feasible=feasible,
    weight=weight,
)
factor_return = e.sorted_factor_return
group_returns = pd.concat(
    [gr.groupby(level=0).mean() for gr in e.group_returns.values()] + [factor_return],
    axis=1,
)
group_returns_mean = group_returns.mean()
group_returns_t = group_returns_mean / (
    group_returns.std() / np.sqrt(group_returns.shape[0])
)
for i in range(len(df) if isinstance(df, list) else 1):
    group_returns_mean.iloc[i * n_groups : (i + 1) * n_groups].plot.bar(
        figsize=(20, 10), title=f"Return for {e._names[i]}"
    )
group_value = (1 + group_returns.shift(1).dropna(how="all", axis=0).fillna(0)).cumprod()
for i in range(len(df) if isinstance(df, list) else 1):
    group_value.iloc[:, i * n_groups : (i + 1) * n_groups].plot(
        figsize=(20, 10), title=f"Group Cumulative Value {e._names[i]}"
    )
group_eval = group_value.apply(quool.Evaluator.evaluate)
display(group_eval)
display(group_returns_t.to_frame("Group Return T Value"))

In [ ]:
e.get_factor_exposure(
    horizon=horizon,
    feasible=feasible,
    window=ts_window,
    min_obs=ts_min_obs,
    intercept=ts_intercept,
    n_jobs=ts_n_jobs,
)
factor_exposure_mean = e.factor_exposure.groupby(level=0).mean()
factor_exposure_mean_t = e.factor_exposure_t.groupby(level=0).mean()
concated = pd.concat(
    [
        factor_exposure_mean,
        e._future_return(horizon).mean().to_frame(f"{horizon}d return"),
    ],
    axis=1,
)
pd.plotting.scatter_matrix(
    concated.iloc[:, 1:],
    figsize=(20, 5 * concated.shape[1] - 1),
    hist_kwds={"bins": 100},
)
concated

In [ ]:
e.cross_sectional_regression(
    horizon=horizon,
    feasible=feasible,
    weight=weight,
    add_intercept=cs_add_intercept,
    cov_type=cs_cov_type,
    white_type=cs_white_type,
)
pd.concat(
    [
        e.factor_premia.iloc[:, 1:],
        e.factor_premia.iloc[:, 1:].cumsum().add_suffix("-Cumsum"),
    ]
).plot(
    figsize=(20, 10),
    secondary_y=(e.factor_premia.columns + "-Cumsum").to_list(),
    title="Factor Return for Each Day & Cumulative Return",
)
pd.concat([e.factor_premia, e.factor_r2, e.factor_premia_t], axis=1)